In [140]:
import pandas as pd
import re

In [ ]:
df = pd.read_excel("DataSample (Ivn_Index).xlsx")
df.head(2)

,wave,question_id,question,answer,answer_count,answer_count_gt_25,respond_gen_id
0,2021-01,q_m5_19_2017_04,[M5] Какой из этих магазинов есть в пешей дост...,Checked,1,0,395189329
1,2021-01,q_m5_19_2017_04,[M5] Какой из этих магазинов есть в пешей дост...,Checked,1,0,395186728


In [142]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56936 entries, 0 to 56935
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   wave                56936 non-null  object
 1   question_id         56936 non-null  object
 2   question            56936 non-null  object
 3   answer              56936 non-null  object
 4   answer_count        56936 non-null  int64 
 5   answer_count_gt_25  56936 non-null  int64 
 6   respond_gen_id      56936 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 3.0+ MB


In [ ]:
# Приводим к нижнему решистру для упрощения работы с текстом
df["question_lower"] = df["question"].str.lower()
df["answer_lower"]   = df["answer"].str.lower()

In [169]:
# Находим все вопросы группы m5 
m5_question = df[df["question_lower"]
                 .str.startswith("[m5]")].copy()
m5_question.head(1)

,wave,question_id,question,answer,answer_count,answer_count_gt_25,respond_gen_id,question_lower,answer_lower
0,2021-01,q_m5_19_2017_04,[M5] Какой из этих магазинов есть в пешей дост...,Checked,1,0,395189329,[m5] какой из этих магазинов есть в пешей дост...,checked


In [ ]:
# Находим магазины которые есть рядом с домом человека 
m5_question["store"] = (
    m5_question["question_lower"]
    .str.split(r"[|&] ", regex=True)
    .str[-1])

In [ ]:
# Нахожим id людей которые живут рядом с пятерочкой 
pyat_id  = m5_question["respond_gen_id"][m5_question["store"]
                                         .str.contains("пятерочка") == 1]
print(f"Количество людей живущих рядом с пятерочкой {len(pyat_id)}")

# Находим id людей которые живут рядом с магнитом 
magnit_id = m5_question["respond_gen_id"][m5_question["store"]
                                          .str.contains("магнит") == 1]
print(f"Количество людей живущих рядом с магнитом {len(magnit_id)}")

In [ ]:
# Находим пересечение двух групп (id людей у которых рядом есть и магнит и пятерочка)
intersection = pyat_id[pyat_id.isin(magnit_id)]
print(f"Количество людей у которых рядом оба магазина = {len(intersection)}")

Количество людей у которых рядом оба магазина = 4960


In [ ]:
# Находим все вопросы группы s8 
s8_question = df[df["question_lower"]
                 .str.startswith("[s8]")].copy()
s8_question.head(2)

,wave,question_id,question,answer,answer_count,answer_count_gt_25,respond_gen_id,question_lower,answer_lower
31,2021-01,q_s8_score_2014_01,[S8] В каком магазине Вы покупали основную час...,Магнит магазин у дома (небольшой формат),58,1,395185753,[s8] в каком магазине вы покупали основную час...,магнит магазин у дома (небольшой формат)
32,2021-01,q_s8_score_2014_01,[S8] В каком магазине Вы покупали основную час...,Магнит магазин у дома (небольшой формат),58,1,395191900,[s8] в каком магазине вы покупали основную час...,магнит магазин у дома (небольшой формат)


In [ ]:
# Находим людей которые чаще всего ходят в пятерочку 
core_pyat_id = pyat_id  = s8_question["respond_gen_id"][s8_question["answer_lower"]
                                         .str.contains("пятерочка") == 1]
print(f"Core аудитория пятерочку = {len(core_pyat_id)}")

2340

In [187]:
# Находим персечение людей у которых рядом с домом два магазина 
# и людей которые чаще ходят в пятерочку (core аудитория)
core_pyat_intersection_id = core_pyat_id[core_pyat_id.isin(intersection)]
print(f"Core аудитория пятерочки у которой рядом есть магнит = {len(core_pyat_intersection_id)}")

Core аудитория пятерочки у которой рядом есть магнит = 1543


In [185]:
# Рассчитываем Conversion rate Пятерочки 
# Conversion rate = количество Core customers пятерочки (в [S8] выбрали пятерочку) /
# / количество респондентов, которые посещают и “Пятерочку”, и “Магнит”
conversion_rate = round(len(core_pyat_intersection_id) / len(intersection), 2)
print(f"Conversion rate Пятерочки = {conversion_rate}")

Conversion rate Пятерочки = 0.31
